# ASL Sign Language Recognition - Google Colab
Upload your project folder to Google Drive, then run this notebook.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Set project path (adjust if your folder is in a different location)
import os
PROJECT_DIR = '/content/drive/MyDrive/HandGestureRecognitionModel'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# Step 3: Install dependencies
!pip install mediapipe==0.10.14 torch torchvision tqdm scikit-learn google-generativeai pyttsx3 python-dotenv -q

In [ ]:
# Step 4: Check GPU availability
import torch
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Step 5: Run Preprocessing (extracts landmarks from videos)
# This is CPU-bound (MediaPipe) but Colab CPUs are faster than most laptops
!python preprocess.py

In [ ]:
# Step 6: Verify processed data
import numpy as np
processed_dir = os.path.join('archive', 'processed_300')
files = [f for f in os.listdir(processed_dir) if f.endswith('.npy')] if os.path.exists(processed_dir) else []
print(f'Processed files: {len(files)}')
if files:
    sample = np.load(os.path.join(processed_dir, files[0]))
    print(f'Sample shape: {sample.shape} (frames, 153 landmarks)')

In [ ]:
# Step 7: Test DataLoader
from dataset import get_dataloader
JSON_PATH = os.path.join('archive', 'nslt_300.json')
DATA_DIR = os.path.join('archive', 'processed_300')

train_loader = get_dataloader(JSON_PATH, DATA_DIR, subset='train', batch_size=16)
print(f'Training batches: {len(train_loader)}')
print(f'Training samples: {len(train_loader.dataset)}')

if len(train_loader) > 0:
    batch_x, batch_y = next(iter(train_loader))
    print(f'Batch shape: {batch_x.shape}')  # Should be (16, 60, 459)

In [ ]:
# Step 8: Train the model (GPU accelerated!)
!python train.py

In [ ]:
# Step 9: Check saved checkpoint
checkpoint_path = os.path.join('checkpoints', 'best_asl_model.pth')
if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024*1024)
    print(f'Model saved! Size: {size_mb:.1f} MB')
    print('Download checkpoints/ folder to use in realtime.py on your laptop.')
else:
    print('No checkpoint found. Training may have failed.')